# Prompt Versioning & Regression Testing — Hands-On

**LLM Engineering · Domain 3 · Roadmap Week 15**

Companion to `02 Literature Notes/LLM Engineering/Prompt Versioning`. Runs fully offline with deterministic fake models.

## 0. Prompt versions are immutable registry records

In [ ]:
%pip install -q numpy
from dataclasses import dataclass
from collections import Counter

@dataclass(frozen=True)
class PromptVersion:
    prompt_id: str
    version: str
    template: str
    model: str
    temperature: float
    eval_set: str
    changelog: str

class Registry:
    def __init__(self): self.items = {}
    def publish(self, p):
        key = (p.prompt_id, p.version)
        if key in self.items: raise ValueError("immutable version already exists")
        self.items[key] = p
    def get(self, prompt_id, version): return self.items[(prompt_id, version)]

reg = Registry()
reg.publish(PromptVersion("support_triage","1.0.0","Classify ticket as billing|bug|other: {ticket}","fake-llm-1",0.0,"gold_v1","initial"))
reg.publish(PromptVersion("support_triage","1.1.0","Classify. Refunds and invoices are billing: {ticket}","fake-llm-1",0.0,"gold_v1","add billing clarification"))
print(reg.get("support_triage","1.1.0"))

## 1. A golden test set encodes expected behavior

In [ ]:
golden = [
    {"id":"g1","input":"Please refund invoice 81","expected":"billing","critical":True},
    {"id":"g2","input":"App crashes with error 500","expected":"bug","critical":True},
    {"id":"g3","input":"How do I change my avatar?","expected":"other","critical":False},
    {"id":"g4","input":"Payment receipt has wrong tax","expected":"billing","critical":False},
]
print("cases", len(golden), "labels", Counter(c["expected"] for c in golden))

## 2. Deterministic fake model: prompt version changes behavior

In [ ]:
def fake_llm(prompt_version, text):
    t = text.lower()
    if "crash" in t or "error" in t:
        return {"label":"bug", "valid_schema":True}
    if "invoice" in t or "payment" in t:
        return {"label":"billing", "valid_schema":True}
    if "refund" in t:
        # v1.0.0 forgot refunds; v1.1.0 fixed it
        fixed = prompt_version.version >= "1.1.0"
        return {"label":"billing" if fixed else "other", "valid_schema":True}
    return {"label":"other", "valid_schema":True}

for v in [reg.get("support_triage","1.0.0"), reg.get("support_triage","1.1.0")]:
    print(v.version, fake_llm(v, "Please refund invoice 81"))

## 3. Regression runner with CI gates

In [ ]:
def evaluate(version, cases):
    rows = []
    for c in cases:
        out = fake_llm(version, c["input"])
        passed = out["valid_schema"] and out["label"] == c["expected"]
        rows.append({**c, "pred":out["label"], "passed":passed})
    acc = sum(r["passed"] for r in rows) / len(rows)
    critical_ok = all(r["passed"] for r in rows if r["critical"])
    return {"version":version.version, "accuracy":acc, "critical_ok":critical_ok, "rows":rows}

for v in [reg.get("support_triage","1.0.0"), reg.get("support_triage","1.1.0")]:
    result = evaluate(v, golden)
    print(result["version"], "accuracy", result["accuracy"], "critical_ok", result["critical_ok"])

## 4. Behavior diffs identify exactly what changed

In [ ]:
base = evaluate(reg.get("support_triage","1.0.0"), golden)
cand = evaluate(reg.get("support_triage","1.1.0"), golden)
for old, new in zip(base["rows"], cand["rows"]):
    if old["pred"] != new["pred"] or old["passed"] != new["passed"]:
        print(old["id"], old["pred"], "->", new["pred"], "expected", new["expected"])
print("candidate passes CI:", cand["accuracy"] >= 0.95 and cand["critical_ok"])

## 5. Model swaps must be evaluated too

In [ ]:
def fake_llm_model_sensitive(prompt_version, text, model):
    out = fake_llm(prompt_version, text)
    if model == "fake-llm-2" and "tax" in text.lower():
        out = {"label":"other", "valid_schema":True}  # regression from a model swap
    return out

def evaluate_model(version, model):
    passed = 0
    for c in golden:
        out = fake_llm_model_sensitive(version, c["input"], model)
        passed += int(out["label"] == c["expected"] and out["valid_schema"])
    return passed / len(golden)

v = reg.get("support_triage","1.1.0")
print("same prompt, model1", evaluate_model(v, "fake-llm-1"))
print("same prompt, model2", evaluate_model(v, "fake-llm-2"), "<- must not silently ship")

## 6. Exercises
1. Add a `cost` column and fail CI if candidate cost rises >20%.
2. Add an adversarial prompt-injection case to the golden set.
3. Implement `bump_version(old, change_type)` for patch/minor/major.
4. Add an A/B report comparing production trace labels by prompt version.

## Links
- Literature note: `02 Literature Notes/LLM Engineering/Prompt Versioning`
- Snippets: `04 Code Snippets/LLM/Prompt Registry With Semantic Versions`, `.../Golden Prompt Regression Harness`
- MOC: `06 Maps of Content/LLM Engineering Concepts`